# Figure P04: 3Di entropy of phage ORFs against bacterial and archaeal ORFs, by length

**Status:** draft, Gadi. The ID follows `figure_plan.md` (#121).

## Provenance

| plotting file | built from | producing stage |
|---|---|---|
| `data/figP04_phage_length_3di.tsv.gz` | Gadi `/g/data/ob80/re3494/inphared_entropy/entropy_3di_results/phg_*.tsv.gz` (6,754,361 per-ORF rows) | agent-handoffs `inphared_gadi/scripts/42_run_entropy.pbs`, `43_extract_phage_rows.py` |
| `data/figP04_gtdb_length_3di.tsv.gz` | Gadi `gtdb_entropy/entropy_null/empirical/length_entropy_{bac,arc}.tsv` (strata 3 = matched and 2 = unmatched in CDS-bearing genomes) | `Validation/gtdb/scripts/33_empirical_length_null.pbs` (#101), full population |
| `data/figP04_matched_density.tsv.gz` | Gadi `inphared_entropy/density/density_{phg,bac,arc}.tsv`, stratum 1 (matched) | agent-handoffs `inphared_gadi/scripts/46_matched_density*` |

- **Reduction:** `manuscript/prep/phage_p04_prepare_data.py`.
  - **Phage:** per-length n, mean and quartiles from the per-ORF rows.
  - **GTDB:** the stage-33 rows copied. Its quantiles are read from 10⁻³-bit histograms, so they are accurate to ±5 × 10⁻⁴ bits.
  - **Densities:** copied as counts.
- **Phage run:**
  - `genome_entropy` 0.3.0 (`v0.3.0`) and `gbouras13/modernprost-50M`, encoding size 10,000, V100;
  - 36,025 Inphared genomes with Pyrodigal-gv/phold annotation;
  - `in_genbank` from the coordinate-anchored matcher, which equals the corrected `v2` phage labels;
  - effective 90 aa floor (issue #103).
- **GTDB run:**
  - `genome_entropy` `v0.2.0` with model revision `2d68d506`, encoding size 10,000, V100;
  - ORF floor 90 aa.
- **Comparability:** the two runs differ in package version. The cross-run determinism test shows per-ORF differences with p99 0.31 bits and class means within about 0.01 bits (Table G01), far smaller than the differences plotted here.
- **Binning:** the 2-D density uses left-closed 0.005-bit bins over 0–4.4 bits, from every matched ORF of each population.


In [ ]:
import gzip
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker

DATA = Path("../data")
IMAGES = Path("../images")
IMAGES.mkdir(exist_ok=True)

# Okabe-Ito, colour-blind safe. One colour per arm throughout the GTDB figures.
ARM = {"candidate": "#0072B2", "comparator": "#D55E00",
       "annotated": "#009E73", "intergenic": "#999999"}
DOMAIN = {"bacteria": "#0072B2", "archaea": "#56B4E9"}
INK = "#222222"
MUTED = "#555555"

plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 7.5,
    "axes.titlesize": 8, "axes.labelsize": 7.5,
    "xtick.labelsize": 7, "ytick.labelsize": 7, "legend.fontsize": 7,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": INK, "axes.labelcolor": INK,
    "xtick.color": INK, "ytick.color": INK,
    "svg.hashsalt": "genome_entropy", "pdf.fonttype": 42,
})


def panel(ax, letter):
    ax.text(-0.02, 1.04, letter, transform=ax.transAxes, fontsize=10,
            fontweight="bold", ha="right", va="bottom")


def save(fig, stem):
    # No timestamps or software strings: reruns are byte-stable.
    fig.savefig(IMAGES / f"{stem}.pdf", bbox_inches="tight",
                metadata={"CreationDate": None, "ModDate": None,
                          "Producer": None, "Creator": None})
    fig.savefig(IMAGES / f"{stem}.png", dpi=300, bbox_inches="tight",
                metadata={"Software": None})
    print("wrote", IMAGES / f"{stem}.pdf", IMAGES / f"{stem}.png")


def matched(path):
    """Long table from 41_comparator_summary.py -> {(domain, metric): (v, lo, hi)}."""
    d = pd.read_csv(path, sep="\t")
    return {(r.domain_name, r.metric): (r.value, r.lo, r.hi) for r in d.itertuples()}


In [ ]:
from matplotlib.colors import LogNorm
ph = pd.read_csv(DATA / "figP04_phage_length_3di.tsv.gz", sep="\t")
gt = pd.read_csv(DATA / "figP04_gtdb_length_3di.tsv.gz", sep="\t")
dn = pd.read_csv(DATA / "figP04_matched_density.tsv.gz", sep="\t")
L = pd.concat([ph, gt], ignore_index=True)
COL = {"phage": "#CC79A7", "bacteria": "#0072B2", "archaea": "#009E73"}
EDGES = np.round(np.arange(0, 881) * 0.005, 4)

fig = plt.figure(figsize=(7.2, 5.4))
gs = fig.add_gridspec(2, 3, height_ratios=[1.15, 1], hspace=0.55, wspace=0.5)
# 40 log-spaced length bins, n-weighted means: per-length phage means are noisy above ~400 aa.
LB = np.unique(np.round(np.geomspace(90, 1000, 41)).astype(int))

ax = fig.add_subplot(gs[0, :2])
for dset in ("bacteria", "archaea", "phage"):
    for cls, ls in (("matched", "-"), ("unmatched", "--")):
        t = L[(L.dataset == dset) & (L.cls == cls)].copy()
        t["b"] = pd.cut(t.aa_length, LB, right=False)
        g = t.assign(w=t["mean"] * t.n).groupby("b", observed=True)
        x = g.aa_length.apply(lambda v: np.exp(np.log(v).mean()))
        y = g.w.sum() / g.n.sum()
        ax.plot(x, y, color=COL[dset], ls=ls, lw=1.0, marker="o", ms=1.6,
                label=f"{dset}, {cls}")
for v in (1.0, np.log2(3)):
    ax.axhline(v, color=MUTED, lw=0.5, ls=(0, (1, 2)), zorder=0)
ax.set_xscale("log")
ax.set_xlim(90, 999)
ax.set_xticks([90, 150, 300, 600, 999], ["90", "150", "300", "600", "999"])
ax.xaxis.set_minor_formatter(matplotlib.ticker.NullFormatter())
ax.set_xlabel("ORF length (aa)")
ax.set_ylabel("mean 3Di entropy (bits)")
ax.set_ylim(0, 3.8)
ax.legend(frameon=False, fontsize=6, ncol=3, loc="lower left", bbox_to_anchor=(0, 1.0))
panel(ax, "a")

ax = fig.add_subplot(gs[0, 2])
BINS = [(90, 149), (150, 299), (300, 599), (600, 999)]
w = 0.38
for k, dset in enumerate(("phage", "bacteria")):
    gaps = []
    for lo, hi in BINS:
        s = {}
        for cls in ("matched", "unmatched"):
            t = L[(L.dataset == dset) & (L.cls == cls) & L.aa_length.between(lo, hi)]
            s[cls] = (t["mean"] * t.n).sum() / t.n.sum()
        gaps.append(s["matched"] - s["unmatched"])
    ax.bar(np.arange(len(BINS)) + (k - 0.5) * w, gaps, width=w, color=COL[dset], label=dset)
ax.set_xticks(range(len(BINS)), [f"{lo}–{hi}" for lo, hi in BINS], rotation=30, ha="right")
ax.set_xlabel("ORF length (aa)")
ax.set_ylabel("matched − unmatched\nmean 3Di entropy (bits)")
ax.legend(frameon=False, fontsize=6, loc="upper left")
panel(ax, "b")

norm = LogNorm(1e-7, 1e-3)
for j, dset in enumerate(("phage", "bacteria", "archaea")):
    ax = fig.add_subplot(gs[1, j])
    d = dn[dn.dataset == dset]
    H = np.zeros((880, 880))
    H[d.protein_bin, d.three_di_bin] = d["count"]
    n = H.sum()
    F = np.where(H > 0, H / n, np.nan)
    m = ax.pcolormesh(EDGES, EDGES, F.T, norm=norm, cmap="viridis", rasterized=True)
    ax.set_xlim(2.4, 4.4)
    ax.set_ylim(0, 4.3)
    ax.set_xlabel("protein entropy (bits)")
    if j == 0:
        ax.set_ylabel("3Di entropy (bits)")
        panel(ax, "c")
    ax.set_title(f"{dset}, matched\nn = {int(n):,}", loc="left", fontsize=6.6, color=COL[dset])
cb = fig.colorbar(m, ax=fig.axes[-3:], fraction=0.02, pad=0.02)
cb.ax.set_title("fraction\nper bin", fontsize=5.5, loc="left")
cb.ax.tick_params(labelsize=5.5)
save(fig, "figP04_phage_vs_cellular")


In [ ]:
rows = []
for dset in ("phage", "bacteria", "archaea"):
    for cls in ("matched", "unmatched"):
        for lo, hi in ((90, 149), (150, 299), (300, 599), (600, 999)):
            t = L[(L.dataset == dset) & (L.cls == cls) & L.aa_length.between(lo, hi)]
            rows.append(dict(dataset=dset, cls=cls, bin=f"{lo}-{hi}", n=int(t.n.sum()),
                             mean=round((t["mean"] * t.n).sum() / t.n.sum(), 3)))
print(pd.DataFrame(rows).pivot_table(index=["bin"], columns=["dataset", "cls"], values="mean").to_string())
print({d: int(dn[dn.dataset == d]["count"].sum()) for d in ("phage", "bacteria", "archaea")})


## Caption (draft)

**Figure P04. Phage ORFs show the same structural separation as cellular ORFs, with matched phage ORFs slightly lower in 3Di entropy at every length.**

**(a)** Mean 3Di entropy by ORF length, from 90 to 999 aa, in 40 log-spaced length bins (n-weighted means of the per-length data). Solid lines are ORFs matched to an annotated CDS, dashed lines unmatched ORFs.
- **Phage:** 36,025 Inphared genomes, 6.75 M ORFs.
- **Bacteria and archaea:** every ORF in CDS-bearing GTDB r232 genomes.
- Matched ORFs gain structural diversity with length while unmatched ORFs lose it, in all three populations.
- Matched phage ORFs sit below matched bacterial ORFs at every length, by 0.24, 0.34, 0.20 and 0.09 bits at 90–149, 150–299, 300–599 and 600–999 aa. The 0.20–0.34 bits quoted in the Inphared Gadi write-up for "≥600 aa" includes ORFs of 1,000 aa and longer. Two explanations are open: phage biology, or label construction (310,360 phage positives match a CDS shorter than their stop-to-stop ORF). This analysis does not separate them.

**(b)** Matched-minus-unmatched mean 3Di entropy in four length ranges. The gap widens with length in both phage and bacteria.

**(c)** Protein against 3Di entropy for **every** matched ORF of each population. Each panel is normalised to its own total, and the populations differ about 115-fold (phage 2.69 M, bacteria 309 M, archaea 9.47 M).

The two runs used different package versions (phage 0.3.0, GTDB 0.2.0) with the same model revision and encoding size. Cross-run differences in class means are about 0.01 bits (Table G01).
